# Domain shift: synthetic-concatenated (S) vs real (R)

- **S (synthetic):** triangles from 1000 training surfaces - the domain we simulate and train on.
- **R (real - KSH):** surface that we want to predict faults on.

Methods:
- **Scaling:** Method 1 (separate scaler per domain), Method 3 (scaler fit on S only), `raw_scaled` - scaling based on the concatenation of S and R.
- **Statististical comparison of features:** per-feature mean, std, Welch t-test, Brown-Forsythe variance test.
- **Visualization:** Distributions (Histograms, Violing, Stripplots). Barplots with coverage. PCA and UMAP; fit on full dataset, plot entire R vs random S subsample of size R.
- **Classifiers:** Check if the the models are able to discriminate between S and R.
- **GMM:** Check if the clusters are correlated with the domain (S/R).

While the single method might be not enough to draw a conclusion, the combination of them gives a big picture of the data. Selected results may be placed in the Paper/Appendix, the rest stay in code.

In [ ]:
from pathlib import Path
from typing import Literal 

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import umap
from IPython.display import display
from matplotlib.figure import Figure
from matplotlib.patches import Patch
from scipy import stats
from sklearn.base import clone, BaseEstimator
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from ss_preprocessing_utils import (
    SORTING_PAIRS,
    SYNTHETIC_PASSTHROUGH_COLS,
    build_sorted_features,
    filter_triangles,
    load_and_filter_real,
    load_raw_synthetic,
    sort_values,
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## Paths and Config

In [ ]:
# Here insert the path to the dataset.
INPUT_PATH = r"../../SubsurfaceBreaks-main/Broken_terrains_datasets"
REAL_DATA_FILE = "KSH_input_output_0.txt"

# Config.
DOMAIN_COL = "Domain"
S_DOMAIN = "S"
R_DOMAIN = "R"
FILE_NUMBER_COL = "File_number"

N_SYNTHETIC_FILES = 1000
RANDOM_STATE = 42
ALPHA = 0.05

OUT_DIR = Path("domain_shift_synthetic_vs_real")
EMBED_PLOT_MATCH_REAL_COUNT = True

# Arbitrary selected values **before** any experiment.
UMAP_NN_GRID = (5, 50, 300)
UMAP_MIN_DIST_GRID = (0.1, 0.5, 0.9)

N_STRIP_POINTS_PER_SERIES = 500
CLASSIFIER_CV_FOLDS = 5

DISTANCE_FAMILIES_N = (
    ("Euclidean_N", ("Euclidean_N_Max", "Euclidean_N_Min", "Euclidean_N_Intermediate")),
    ("Cosine_N", ("Cosine_N_Max", "Cosine_N_Min", "Cosine_N_Intermediate")),
    ("Angle_N", ("Angle_N_Max", "Angle_N_Min", "Angle_N_Intermediate")),
)
DISTANCE_FAMILIES_D = (
    ("Euclidean_D", ("Euclidean_D_Max", "Euclidean_D_Min", "Euclidean_D_Intermediate")),
    ("Cosine_D", ("Cosine_D_Max", "Cosine_D_Min", "Cosine_D_Intermediate")),
    ("Angle_D", ("Angle_D_Max", "Angle_D_Min", "Angle_D_Intermediate")),
)

COORDS_N = ("X_N", "Y_N", "Z_N")
COORDS_D = ("X_D", "Y_D", "Z_D")

SERIES_LABELS = ("S_Max", "S_Min", "S_Int", "R_Max", "R_Min", "R_Int")
SERIES_COLORS = (
    "#10B981",
    "#1D4ED8",
    "#0F766E",
    "#6D28D9",
    "#F59E0B",
    "#64748B",
)

SVC_MODEL_PARAMS = {"kernel": "rbf", "C": 10, "gamma": 0.1}

In [ ]:
# Execute config.
np.random.seed(RANDOM_STATE)
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Preprocessing flow

In [ ]:
def load_domains(
    input_path: str,
    real_file: str,
    n_synthetic_files: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, list[str]]:
    """Load synthetic (S) and real (R) domains,
    perform standard preprocessing flow as in the other experiments.
    """
    raw_s = load_raw_synthetic(input_path, n_synthetic_files)
    filtered_s = filter_triangles(raw_s)
    # filtered_s.to_csv("s_filtered_not_sorted.csv", index=False)
    sorted_s = build_sorted_features(
        filtered_s,
        SORTING_PAIRS,
        sort_values,
        SYNTHETIC_PASSTHROUGH_COLS,
    )
    sorted_s = pd.concat(
        [sorted_s, filtered_s[[FILE_NUMBER_COL]].reset_index(drop=True)],
        axis=1,
    )
    sorted_s[DOMAIN_COL] = S_DOMAIN
    # sorted_s.to_csv("s_filtered_sorted_for_analysis.csv", index=False)

    raw_r = load_and_filter_real(input_path, real_file)
    sorted_r = build_sorted_features(
        raw_r,
        SORTING_PAIRS,
        sort_values,
        SYNTHETIC_PASSTHROUGH_COLS,
    )
    sorted_r[FILE_NUMBER_COL] = -1
    sorted_r[DOMAIN_COL] = R_DOMAIN

    feature_cols = [c for c in sorted_s.columns if c not in (FILE_NUMBER_COL, DOMAIN_COL)]
    all_data = pd.concat(
        [
            sorted_s[feature_cols + [DOMAIN_COL, FILE_NUMBER_COL]],
            sorted_r[feature_cols + [DOMAIN_COL, FILE_NUMBER_COL]],
        ],
        ignore_index=True,
    )
    return sorted_s, sorted_r, all_data, feature_cols

(
    sorted_s,
    sorted_r,
    all_data,
    feature_cols,
) = load_domains(
    INPUT_PATH,
    REAL_DATA_FILE,
    N_SYNTHETIC_FILES,
)

n_s_triangles = len(sorted_s)
n_r_triangles = len(sorted_r)
figure_title_suffix = f"Full data (n_S={n_s_triangles}, n_R={n_r_triangles})"
print(
    f"Synthetic triangles: {n_s_triangles:,} | "
    f"Real triangles: {n_r_triangles:,} | "
    f"Features: {len(feature_cols)}"
)

In [ ]:
# sorted_r.to_csv("real_sorted.csv", index=False)

## Scaling

In [ ]:
# Helper functions.
def fit_scale_method1(
    df_s: pd.DataFrame, df_r: pd.DataFrame, feature_cols: list[str]
) -> tuple[np.ndarray, np.ndarray, StandardScaler, StandardScaler]:
    """Fit separate scalers for S and R."""
    scaler_s = StandardScaler().fit(df_s[feature_cols])
    scaler_r = StandardScaler().fit(df_r[feature_cols])
    x_s = scaler_s.transform(df_s[feature_cols])
    x_r = scaler_r.transform(df_r[feature_cols])
    return x_s, x_r, scaler_s, scaler_r


def fit_scale_method3(
    df_s: pd.DataFrame, df_r: pd.DataFrame, feature_cols: list[str]
) -> tuple[np.ndarray, np.ndarray, StandardScaler]:
    """Fit a single scaler on the S and transform both S and R."""
    scaler = StandardScaler().fit(df_s[feature_cols])
    x_s = scaler.transform(df_s[feature_cols])
    x_r = scaler.transform(df_r[feature_cols])
    return x_s, x_r, scaler


def fit_raw_scaled(
    all_data: pd.DataFrame, feature_cols: list[str]
) -> tuple[np.ndarray, np.ndarray, StandardScaler]:
    """Fit a single scaled on the full dataset.
    Returns the transformed S and R separately.
    """
    scaler = StandardScaler().fit(all_data[feature_cols])
    mask_s = all_data[DOMAIN_COL] == S_DOMAIN
    x_s = scaler.transform(all_data.loc[mask_s, feature_cols])
    x_r = scaler.transform(all_data.loc[~mask_s, feature_cols])
    return x_s, x_r, scaler


def scaled_frames_from_result_arrays(
    feature_cols: list[str],
    x_s: np.ndarray,
    x_r: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    scaled_s = pd.DataFrame(x_s, columns=feature_cols)
    scaled_r = pd.DataFrame(x_r, columns=feature_cols)
    return scaled_s, scaled_r

In [ ]:
method1_s_array, method1_r_array, scaler_s_m1, scaler_r_m1 = fit_scale_method1(
    sorted_s, sorted_r, feature_cols
)
method1_s_frame, method1_r_frame = scaled_frames_from_result_arrays(
    feature_cols,
    method1_s_array,
    method1_r_array,
)

method3_s_array, method3_r_array, scaler_m3 = fit_scale_method3(
    sorted_s, sorted_r, feature_cols
)
method3_s_frame, method3_r_frame = scaled_frames_from_result_arrays(
    feature_cols,
    method3_s_array,
    method3_r_array,
)

raw_scaled_s_array, raw_scaled_r_array, scaler_raw_scaled = fit_raw_scaled(
    all_data, feature_cols
)

raw_scaled_s_frame, raw_scaled_r_frame = scaled_frames_from_result_arrays(
    feature_cols,
    raw_scaled_s_array,
    raw_scaled_r_array,
)

## Statistics

In [ ]:
# Helper functions.
def brown_forsythe_pvalue(s_values: np.ndarray, r_values: np.ndarray) -> float:
    """Brown-Forsythe test for equality of variances (two groups)."""
    # Ensure proper types.
    s_values = np.asarray(s_values, dtype=float)
    r_values = np.asarray(r_values, dtype=float)
    # Remove invalid values.
    s_values = s_values[np.isfinite(s_values)]
    r_values = r_values[np.isfinite(r_values)]
    # Perform the Brown-Forsythe test.
    # Note: stats.levene(..., center='median') is the same as the Brown-Forsythe test.
    _, p_value = stats.levene(s_values, r_values, center='median')
    return float(p_value)

def comparison_table_statistics(
    df_s: pd.DataFrame,
    df_r: pd.DataFrame,
    feature_cols: list[str],
) -> pd.DataFrame:
    """Return a table with means, stds, Welch and Brown-Forsythe p-values.
    The p-value shouldn't be interpreted alone without the other statistics
    due to significant sample size difference between the two groups.
    """
    rows = []
    for col in feature_cols:
        s_vals = df_s[col].to_numpy(dtype=float)
        r_vals = df_r[col].to_numpy(dtype=float)
        # Welch's t-test - equal_var=False.
        welch = stats.ttest_ind(s_vals, r_vals, equal_var=False)
        rows.append(
            {
                "STATISTICAL VARIABLE": col,
                "S_mean": float(np.mean(s_vals)),
                "S_std": float(np.std(s_vals, ddof=1)),
                "R_mean": float(np.mean(r_vals)),
                "R_std": float(np.std(r_vals, ddof=1)),
                "n_S": int(len(s_vals)),
                "n_R": int(len(r_vals)),
                "welch_p-value": float(welch.pvalue),
                "brown_forsythe_p-value": brown_forsythe_pvalue(s_vals, r_vals),
            }
        )
    return pd.DataFrame(rows)


def transformation_parameters_table(
    feature_cols: list[str],
    scaler_m3: StandardScaler,
    scaler_s_m1: StandardScaler,
    scaler_r_m1: StandardScaler,
    scaler_raw_scaled: StandardScaler,
) -> pd.DataFrame:
    """Long-form scaler parameters for method1, method3, raw_scaled."""
    rows = []
    for i, var in enumerate(feature_cols):
        rows.append(
            {
                "STATISTICAL VARIABLE": var,
                "scaling": "method3",
                "M3_scaler_mean": float(scaler_m3.mean_[i]),
                "M3_scaler_scale": float(scaler_m3.scale_[i]),
                "M1_S_scaler_mean": np.nan,
                "M1_S_scaler_scale": np.nan,
                "M1_R_scaler_mean": np.nan,
                "M1_R_scaler_scale": np.nan,
                "RAW_SCALED_scaler_mean": np.nan,
                "RAW_SCALED_scaler_scale": np.nan,
            }
        )
        rows.append(
            {
                "STATISTICAL VARIABLE": var,
                "scaling": "method1",
                "M3_scaler_mean": np.nan,
                "M3_scaler_scale": np.nan,
                "M1_S_scaler_mean": float(scaler_s_m1.mean_[i]),
                "M1_S_scaler_scale": float(scaler_s_m1.scale_[i]),
                "R_scaler_mean": float(scaler_r_m1.mean_[i]),
                "R_scaler_scale": float(scaler_r_m1.scale_[i]),
                "pooled_scaler_mean": np.nan,
                "pooled_scaler_scale": np.nan,
            }
        )
        rows.append(
            {
                "STATISTICAL VARIABLE": var,
                "scaling": "raw_scaled",
                "M3_scaler_mean": np.nan,
                "M3_scaler_scale": np.nan,
                "M1_S_scaler_mean": np.nan,
                "M1_S_scaler_scale": np.nan,
                "M1_R_scaler_mean": np.nan,
                "M1_R_scaler_scale": np.nan,
                "RAW_SCALED_scaler_mean": float(scaler_raw_scaled.mean_[i]),
                "RAW_SCALED_scaler_scale": float(scaler_raw_scaled.scale_[i]),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
comparison_table_raw_features = comparison_table_statistics(
    sorted_s, sorted_r, feature_cols
)
comparison_table_method1_scaled = comparison_table_statistics(
    method1_s_frame, method1_r_frame, feature_cols
)
comparison_table_method3_scaled = comparison_table_statistics(
    method3_s_frame, method3_r_frame, feature_cols
)

In [ ]:
# Equal size
sorted_s_equal_size = sorted_s.sample(n=n_r_triangles, random_state=RANDOM_STATE)
method1_s_frame_equal_size = method1_s_frame.loc[sorted_s_equal_size.index]
method3_s_frame_equal_size = method3_s_frame.loc[sorted_s_equal_size.index]

comparison_table_raw_features_equal_size = comparison_table_statistics(
    sorted_s_equal_size, sorted_r, feature_cols
)
comparison_table_method1_scaled_equal_size = comparison_table_statistics(
    method1_s_frame_equal_size, method1_r_frame, feature_cols
)
comparison_table_method3_scaled_equal_size = comparison_table_statistics(
    method3_s_frame_equal_size, method3_r_frame, feature_cols
)

In [ ]:
transformation_parameters = transformation_parameters_table(
    feature_cols,
    scaler_m3,
    scaler_s_m1,
    scaler_r_m1,
    scaler_raw_scaled,
)

In [ ]:
# Helper functions.
def export_comparison_table_graphic(
    table: pd.DataFrame,
    method_name: str, 
    out_path: Path
) -> None:
    """Export a comparison table graphic."""
    display_cols = [
        "STATISTICAL VARIABLE",
        "n_S",
        "S_mean",
        "S_std",
        "n_R",
        "R_mean",
        "R_std",
        "welch_p-value",
        "brown_forsythe_p-value",
    ]
    show = table[display_cols].copy()
    for col in ("S_mean", "S_std", "R_mean", "R_std"):
        show[col] = show[col].map(lambda v: f"{v:.3f}")
    for col in ("welch_p-value", "brown_forsythe_p-value"):
        show[col] = show[col].map(lambda v: f"{v:.7f}")

    fig_h = max(9, 0.28 * len(show))
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis("off")
    tbl = ax.table(
        cellText=show.values,
        colLabels=show.columns,
        loc="center",
        cellLoc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(6)
    tbl.scale(1, 1.2)
    fig.suptitle(f"Method: {method_name}", fontsize=10, y=0.98)
    fig.tight_layout()

    fig.savefig(
        out_path,
        format="svg",
        bbox_inches="tight",
        dpi=1200,
    )
    plt.close(fig)

def write_comparison_table_outputs(
    frame: pd.DataFrame,
    method_name: str,
) -> None:
    """Write a comparison table to a csv and png file."""
    csv_path = OUT_DIR.stem + "/statistical_comparison_" + method_name + '.csv'
    frame.to_csv(csv_path, index=False)
    png_path = OUT_DIR.stem + "/statistical_comparison_" + method_name + '.svg'
    export_comparison_table_graphic(frame, method_name, png_path)

In [ ]:
write_comparison_table_outputs(comparison_table_raw_features, "raw")
write_comparison_table_outputs(comparison_table_method1_scaled, "method_1")
write_comparison_table_outputs(comparison_table_method3_scaled, "method_3")
write_comparison_table_outputs(comparison_table_raw_features_equal_size, "raw_equal_size")
write_comparison_table_outputs(comparison_table_method1_scaled_equal_size, "method_1_equal_size")
write_comparison_table_outputs(comparison_table_method3_scaled_equal_size, "method_3_equal_size")
trans_path = OUT_DIR / "table_transformation_parameters.csv"
transformation_parameters.to_csv(trans_path, index=False)

## Distributions

In [ ]:
# General helper function.
def _series_arrays(
    df_s: pd.DataFrame, df_r: pd.DataFrame, cols: list[str]
) -> list[np.ndarray]:
    """Convert a list of columns from a dataframe to a list of numpy arrays."""
    s_arrays = [df_s[col].to_numpy(dtype=float) for col in cols]
    r_arrays = [df_r[col].to_numpy(dtype=float) for col in cols]
    return s_arrays + r_arrays

In [ ]:
# Histogram + kde functions.
def _histogram_kde_per_ax(
    ax: plt.Axes, 
    arrays: list[np.ndarray], 
    labels: list[str], 
    colors: list[str], 
    title: str, 
    apply_symlog: bool = False
) -> None:
    """Plot an aligned histogram and KDE for a family of features."""
    global_min = float('inf')
    global_max = float('-inf')
    valid_arrays = []
    
    for array in arrays:
        array = array[~np.isnan(array)]
        valid_arrays.append(array)
        global_min = min(global_min, array.min())
        global_max = max(global_max, array.max())

    for array, label, color in zip(valid_arrays, labels, colors, strict=False):
        ax.hist(array, bins=40, alpha=0.4, density=True, color=color, label=label, edgecolor="none")
        sns.kdeplot(array, ax=ax, color=color, linewidth=1.5, label=f"{label} KDE", 
                    clip=(global_min, global_max), bw_adjust=3.0, cut=0)

    title_suffix = ""
    if apply_symlog:
        ax.set_yscale('symlog', linthresh=0.1) 
        title_suffix = " (Y: symlog)"

    ax.set_title(f"{title}{title_suffix}", fontsize=9)
    ax.set_xlim(global_min - 0.05 * abs(global_min-global_max), global_max + 0.05 * abs(global_max-global_min))
    ax.set_ylabel("") 
    ax.legend(fontsize=6, loc="upper right", ncol=1)


def plot_histogram(
    df_s: pd.DataFrame,
    df_r: pd.DataFrame,
    apply_symlog: bool,
) -> Figure:
    """Create histogram for visual comparison of features between S and R."""
    fig, axes = plt.subplots(4, 3, figsize=(26, 13.5))

    coords_labels = [S_DOMAIN, R_DOMAIN]
    coords_colors = ["#1f4e79", "#c55a11"]

    for col_idx, (family_name, cols) in enumerate(DISTANCE_FAMILIES_N):
        arrays = _series_arrays(df_s, df_r, cols)
        _histogram_kde_per_ax(axes[0, col_idx], arrays, SERIES_LABELS , SERIES_COLORS, family_name, apply_symlog=apply_symlog)
        
    for col_idx, (family_name, cols) in enumerate(DISTANCE_FAMILIES_D):
        arrays = _series_arrays(df_s, df_r, cols)
        _histogram_kde_per_ax(axes[1, col_idx], arrays, SERIES_LABELS , SERIES_COLORS, family_name, apply_symlog=apply_symlog)
        
    for col_idx, feature_name in enumerate(COORDS_N):
        arrays = (df_s[feature_name].to_numpy(), df_r[feature_name].to_numpy())
        _histogram_kde_per_ax(axes[2, col_idx], arrays, coords_labels, coords_colors, feature_name, apply_symlog=False)
        
    for col_idx, feature_name in enumerate(COORDS_D):
        arrays = (df_s[feature_name].to_numpy(), df_r[feature_name].to_numpy())
        _histogram_kde_per_ax(axes[3, col_idx], arrays, coords_labels, coords_colors, feature_name, apply_symlog=False)

    fig.suptitle(f"Feature histograms + KDE - {figure_title_suffix}", fontsize=14, y=0.98)
    fig.tight_layout()
    return fig

In [ ]:
histogram_fig_raw = plot_histogram(
    sorted_s,
    sorted_r,
    apply_symlog=True,
)
histogram_fig_m1 = plot_histogram(
    method1_s_frame,
    method1_r_frame,
    apply_symlog=False,
)
histogram_fig_m3 = plot_histogram(
    method3_s_frame,
    method3_r_frame,
    apply_symlog=False,
)

histogram_fig_raw.savefig(
    OUT_DIR / "fig_histograms_raw.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)
histogram_fig_m1.savefig(
    OUT_DIR / "fig_histograms_m1.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)
histogram_fig_m3.savefig(
    OUT_DIR / "fig_histograms_m3.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)

In [ ]:
# Violin plot + stripplot functions.
def _plot_violin_with_strip(
    ax: plt.Axes, 
    df: pd.DataFrame, 
    palette: list[str], 
    title: str, 
    n_strip: int, 
    random_state: int
) -> None:
    """Helper function to plot a violin plot and a stripplot overlay on a given axis."""
    categories = df["series"].unique()
    
    sns.violinplot(
        data=df, 
        x="series", 
        y="value", 
        ax=ax, 
        inner=None, 
        palette=palette,
        hue="series",
        order=categories,
        legend=False
    )
    
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("")
    ax.set_ylabel("")

    rng = np.random.default_rng(random_state)
    for i, series in enumerate(categories):
        sub = df.loc[df["series"] == series, "value"].dropna()
        if len(sub) == 0:
            continue
            
        n_take = min(n_strip, len(sub))
        strip = sub.sample(n=n_take, random_state=random_state)
        jitter = rng.uniform(-0.15, 0.15, size=len(strip))
        
        ax.scatter(i + jitter, strip, s=1, alpha=0.15, c="k", zorder=3)
        
    ax.tick_params(axis="x", rotation=45, labelsize=7)


def plot_violin_strip(
    df_s: pd.DataFrame,
    df_r: pd.DataFrame,
    n_strip: int,
    random_state: int,
) -> Figure:
    """Build the entire violin plot + stripplot."""
    fig, axes = plt.subplots(4, 3, figsize=(26, 13.5))

    half_colors = len(SERIES_COLORS) // 2
    dist_palette = []
    for i in range(half_colors):
        dist_palette.extend([SERIES_COLORS[i], SERIES_COLORS[i + half_colors]])
    
    coord_palette = ["#1f4e79", "#c55a11"]

    for row_idx, dist_families in enumerate([DISTANCE_FAMILIES_N, DISTANCE_FAMILIES_D]):
        for col_idx, (family_name, cols) in enumerate(dist_families):
            dfs = []
            for col, slabel in zip(cols, SERIES_LABELS[:3], strict=True):
                suffix = slabel.split('_')[1]
                dfs.append(pd.DataFrame({"value": df_s[col], "series": f"{S_DOMAIN}_{suffix}"}))
                dfs.append(pd.DataFrame({"value": df_r[col], "series": f"{R_DOMAIN}_{suffix}"}))
                
            df_long = pd.concat(dfs, ignore_index=True)
            _plot_violin_with_strip(axes[row_idx, col_idx], df_long, dist_palette, family_name, n_strip, random_state)

    for row_idx, coords in enumerate([COORDS_N, COORDS_D], start=2):
        for col_idx, feature_name in enumerate(coords):
            df_long = pd.concat([
                pd.DataFrame({"value": df_s[feature_name], "series": S_DOMAIN}),
                pd.DataFrame({"value": df_r[feature_name], "series": R_DOMAIN})
            ], ignore_index=True)
            
            _plot_violin_with_strip(axes[row_idx, col_idx], df_long, coord_palette, feature_name, n_strip, random_state)

    fig.suptitle(
        f"Violin + stripplot - stripplot n={n_strip} per series; {figure_title_suffix}",
        fontsize=14, y=0.98
    )
    fig.tight_layout()
    
    return fig

In [ ]:
violin_plot_fig_raw = plot_violin_strip(
    sorted_s,
    sorted_r,
    N_STRIP_POINTS_PER_SERIES,
    RANDOM_STATE,
)
violin_plot_fig_m1 = plot_violin_strip(
    method1_s_frame,
    method1_r_frame,
    N_STRIP_POINTS_PER_SERIES,
    RANDOM_STATE,
)
violin_plot_fig_m3 = plot_violin_strip(
    method3_s_frame,
    method3_r_frame,
    N_STRIP_POINTS_PER_SERIES,
    RANDOM_STATE,
)

violin_plot_fig_raw.savefig(
    OUT_DIR / "fig_violins_raw.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)
violin_plot_fig_m1.savefig(
    OUT_DIR / "fig_violins_m1.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)
violin_plot_fig_m3.savefig(
    OUT_DIR / "fig_violins_m3.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)

## Coverage

In [ ]:
# Coverage plot functions.
def feature_range_coverage(
    df_s: pd.DataFrame,
    df_r: pd.DataFrame,
    feature_cols: list[str],
) -> pd.DataFrame:
    """Calculate actual ranges and overlap."""
    rows = []
    for col in feature_cols:
        s_min, s_max = float(df_s[col].min()), float(df_s[col].max())
        r_min, r_max = float(df_r[col].min()), float(df_r[col].max())
        overlap_min = max(s_min, r_min)
        overlap_max = min(s_max, r_max)
        overlap_width = max(0.0, overlap_max - overlap_min)
        
        rows.append(
            {
                "variable": col,
                "S_min": s_min,
                "S_max": s_max,
                "R_min": r_min,
                "R_max": r_max,
                "Overlap_min": overlap_min if overlap_max > overlap_min else np.nan,
                "Overlap_max": overlap_max if overlap_max > overlap_min else np.nan,
                "Overlap_width": overlap_width
            }
        )
    return pd.DataFrame(rows)


def _draw_segmented_bar(
    ax: plt.Axes, 
    y: float, 
    v_min: float, 
    v_max: float, 
    global_min: float, 
    global_max: float, 
    color: str, 
    height: float = 0.5
):
    """Helper for bars."""
    if v_max > v_min:
        ax.barh(y, v_max - v_min, left=v_min, height=height, color=color, edgecolor="none")
    if v_min > global_min:
        ax.barh(y, v_min - global_min, left=global_min, height=height, color="black", edgecolor="none")
    if v_max < global_max:
        ax.barh(y, global_max - v_max, left=v_max, height=height, color="black", edgecolor="none")


def plot_feature_coverage(
    df_s: pd.DataFrame,
    df_r: pd.DataFrame,
    feature_cols: list[str],
) -> tuple[Figure, pd.DataFrame]:
    """Create a bar plot representing actual feature ranges
    and zones not reflected in the second domain.
    """
    cov = feature_range_coverage(df_s, df_r, feature_cols)
    
    n_features = len(feature_cols)
    fig, axes = plt.subplots(nrows=n_features, ncols=1, figsize=(13, max(4.5, n_features * 0.75)))

    color_s = "#1f4e79"
    color_r = "#c55a11"
    
    for ax, (_, row) in zip(axes, cov.iterrows(), strict=True):
        s_min, s_max = row["S_min"], row["S_max"]
        r_min, r_max = row["R_min"], row["R_max"]
        
        global_min = min(s_min, r_min)
        global_max = max(s_max, r_max)
        
        _draw_segmented_bar(ax, y=1, v_min=s_min, v_max=s_max, global_min=global_min, global_max=global_max, color=color_s)
        _draw_segmented_bar(ax, y=0, v_min=r_min, v_max=r_max, global_min=global_min, global_max=global_max, color=color_r)
        
        ax.set_yticks([0, 1])
        ax.set_yticklabels(["R", "S"], fontsize=8)
        ax.set_ylabel(row["variable"], rotation=0, labelpad=20, ha='right', va='center', fontweight='bold', fontsize=9)
        
        margin = (global_max - global_min) * 0.05
        ax.set_xlim(global_min - margin, global_max + margin)
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.grid(axis='x', linestyle='--', alpha=0.4)

    legend_elements = [
        Patch(facecolor=color_s, label='S range'),
        Patch(facecolor=color_r, label='R range'),
        Patch(facecolor='black', label='Missing observations (Lack of overlap)')
    ]
    
    fig.legend(handles=legend_elements, loc='lower center', ncol=3, bbox_to_anchor=(0.5, 0.94))
    fig.suptitle(f"Feature Range Coverage - {figure_title_suffix}", y=0.99, fontsize=12)
    fig.align_ylabels(axes)
    
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    
    return fig, cov

In [ ]:
coverage_fig_raw, coverage_table_raw = plot_feature_coverage(
    sorted_s,
    sorted_r,
    feature_cols,
)
coverage_fig_m1, coverage_table_m1 = plot_feature_coverage(
    method1_s_frame,
    method1_r_frame,
    feature_cols,
)
coverage_fig_m3, coverage_table_m3 = plot_feature_coverage(
    method3_s_frame,
    method3_r_frame,
    feature_cols,
)

coverage_table_raw.to_csv(OUT_DIR / "feature_range_coverage_raw.csv", index=False)
coverage_fig_raw.savefig(
    OUT_DIR / "fig_feature_coverage_raw.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)
coverage_table_m1.to_csv(OUT_DIR / "feature_range_coverage_m1.csv", index=False)
coverage_fig_m1.savefig(
    OUT_DIR / "fig_feature_coverage_m1.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)
coverage_table_m3.to_csv(OUT_DIR / "feature_range_coverage_m3.csv", index=False)
coverage_fig_m3.savefig(
    OUT_DIR / "fig_feature_coverage_m3.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)

## PCA and UMAP

In [ ]:
# Helper functions.
def select_indices(
    n_s: int, n_r: int, random_state: int
) -> tuple[np.ndarray, np.ndarray]:
    """Select up to n_r random indices from S domain 
    and generate sequential indices for R domain.
    Return both indices arrays.
    """
    idx_r = np.arange(n_r, dtype=int)
    rng = np.random.default_rng(random_state)
    n_plot_s = min(n_r, n_s)
    idx_s = rng.choice(n_s, size=n_plot_s, replace=False)
    return idx_s, idx_r


def _concat_domains(x_s: np.ndarray, x_r: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Concat two domains.
    S domain has label 0, R domain has label 1.
    Return concatenated data and labels.
    """
    labels = np.concatenate(
        [np.zeros(len(x_s), dtype=int), np.ones(len(x_r), dtype=int)]
    )
    x = np.vstack([x_s, x_r])
    return x, labels

def _scatter_plot_coords(
    ax: plt.Axes, coords_s: np.ndarray, coords_r: np.ndarray, reversed_: bool
) -> None:
    if reversed_:
        ax.scatter(coords_r[:, 0], coords_r[:, 1], s=2, alpha=0.3, label="R", c="#c55a11")
        ax.scatter(coords_s[:, 0], coords_s[:, 1], s=2, alpha=0.3, label="S", c="#1f4e79")
    else:
        ax.scatter(coords_s[:, 0], coords_s[:, 1], s=2, alpha=0.3, label="S", c="#1f4e79")
        ax.scatter(coords_r[:, 0], coords_r[:, 1], s=2, alpha=0.3, label="R", c="#c55a11")
    ax.legend(markerscale=3, fontsize=7)

In [ ]:
# Plot PCA function.
def plot_pca(
    x_s: np.ndarray,
    x_r: np.ndarray,
    random_state: int,
) -> Figure:
    """Create PCA plot."""
    n_s = len(x_s)
    n_r = len(x_r)
    idx_s, idx_r = select_indices(n_s, n_r, random_state)
    n_s_plot = len(idx_s)

    x_pool, _ = _concat_domains(x_s, x_r)
    pca = PCA(n_components=2, random_state=random_state)
    coords_pool = pca.fit_transform(x_pool)
    coords_pool_s = coords_pool[:n_s][idx_s]
    coords_pool_r = coords_pool[n_s:][idx_r]

    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    _scatter_plot_coords(ax[0], coords_pool_s, coords_pool_r, False)
    ax[0].set_title("Order: plot S then R")
    _scatter_plot_coords(ax[1], coords_pool_s, coords_pool_r, True)
    ax[1].set_title("Order: plot R then S")
    fig.suptitle(
        f"PCA - fit: {figure_title_suffix};"
        f"\nplot: n_S_plot={n_s_plot}, n_R_plot={n_r}, seed={random_state}",
    )
    fig.tight_layout()
    return fig

# Plot UMAP function.
def plot_umap_grid(
    x_s: np.ndarray,
    x_r: np.ndarray,
    random_state: int,
    umap_neighbors: tuple[int, ...],
    umap_min_dist: tuple[float, ...],
    reversed_: bool = False,
) -> Figure:
    """Create UMAP plot."""
    n_s = len(x_s)
    n_r = len(x_r)
    idx_s, idx_r = select_indices(n_s, n_r, random_state)
    n_s_plot = len(idx_s)

    x_pool, _ = _concat_domains(x_s, x_r)

    nrows = len(umap_neighbors)
    ncols = len(umap_min_dist)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 5*nrows))

    for i, nn in enumerate(umap_neighbors):
        for j, md in enumerate(umap_min_dist):
            ax = axes[i, j]
            embeder = umap.UMAP(
                n_neighbors=nn,
                min_dist=md,
                n_components=2,
                random_state=random_state,
                n_jobs=-1,
            )

            coords_pool = embeder.fit_transform(x_pool)
            
            coords_pool_s = coords_pool[:n_s][idx_s]
            coords_pool_r = coords_pool[n_s:][idx_r]

            _scatter_plot_coords(ax, coords_pool_s, coords_pool_r, reversed_)
            ax.set_title(f"nn={nn}, min_dist={md}", fontsize=8)

    first = "R" if reversed_ else "S"
    second = "S" if reversed_ else "R"
    fig.suptitle(
        f"UMAP - fit: {figure_title_suffix}; Order: plot {first} then {second}"
        f"\nplot: n_S_plot={n_s_plot}, n_R_plot={n_r}, seed={random_state}",
    )
    fig.tight_layout()
    return fig

In [ ]:
pca_plot_fig_raw = plot_pca(
    raw_scaled_s_frame,
    raw_scaled_r_frame,
    RANDOM_STATE,
)
pca_plot_fig_m1 = plot_pca(
    method1_s_frame,
    method1_r_frame,
    RANDOM_STATE,
)
pca_plot_fig_m3 = plot_pca(
    method3_s_frame,
    method3_r_frame,
    RANDOM_STATE,
)

pca_plot_fig_raw.savefig(
    OUT_DIR / "fig_pca_raw_scaled.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)
pca_plot_fig_m1.savefig(
    OUT_DIR / "fig_pca_m1.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)
pca_plot_fig_m3.savefig(
    OUT_DIR / "fig_pca_m3.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)

In [ ]:
umap_plot_fig_raw = plot_umap_grid(
    raw_scaled_s_frame,
    raw_scaled_r_frame,
    RANDOM_STATE,
    UMAP_NN_GRID,
    UMAP_MIN_DIST_GRID,
    reversed_=False,
)
umap_plot_fig_raw.savefig(
    OUT_DIR / "fig_umap_grid_raw_scaled.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)
plt.close(umap_plot_fig_raw)

umap_plot_fig_m1 = plot_umap_grid(
    method1_s_frame,
    method1_r_frame,
    RANDOM_STATE,
    UMAP_NN_GRID,
    UMAP_MIN_DIST_GRID,
    reversed_=False,
)
umap_plot_fig_m1.savefig(
    OUT_DIR / "fig_umap_grid_m1.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)
plt.close(umap_plot_fig_m1)

umap_plot_fig_m3 = plot_umap_grid(
    method3_s_frame,
    method3_r_frame,
    RANDOM_STATE,
    UMAP_NN_GRID,
    UMAP_MIN_DIST_GRID,
    reversed_=False,
)
umap_plot_fig_m3.savefig(
    OUT_DIR / "fig_umap_grid_m3.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)
plt.close(umap_plot_fig_m3)

umap_plot_fig_raw_reversed = plot_umap_grid(
    raw_scaled_s_frame,
    raw_scaled_r_frame,
    RANDOM_STATE,
    UMAP_NN_GRID,
    UMAP_MIN_DIST_GRID,
    reversed_=True,
)
umap_plot_fig_raw_reversed.savefig(
    OUT_DIR / "fig_umap_grid_reversed_raw_scaled.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)
plt.close(umap_plot_fig_raw_reversed)

umap_plot_fig_m1_reversed = plot_umap_grid(
    method1_s_frame,
    method1_r_frame,
    RANDOM_STATE,
    UMAP_NN_GRID,
    UMAP_MIN_DIST_GRID,
    reversed_=True,
)
umap_plot_fig_m1_reversed.savefig(
    OUT_DIR / "fig_umap_grid_reversed_m1.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)
plt.close(umap_plot_fig_m1_reversed)

umap_plot_fig_m3_reversed = plot_umap_grid(
    method3_s_frame,
    method3_r_frame,
    RANDOM_STATE,
    UMAP_NN_GRID,
    UMAP_MIN_DIST_GRID,
    reversed_=True,
)
umap_plot_fig_m3_reversed.savefig(
    OUT_DIR / "fig_umap_grid_reversed_m3.svg",
    format="svg",
    bbox_inches="tight",
    facecolor="white",
    dpi=1200,
)
plt.close(umap_plot_fig_m3_reversed)

# Classifiers

In [ ]:
def _apply_scaling(
    X_train: np.ndarray, 
    X_test: np.ndarray, 
    is_s_train: np.ndarray, 
    is_s_test: np.ndarray, 
    method: Literal["raw", "method1a", "method1b", "method1c", "method3a", "method3b"],
) -> tuple[np.ndarray, np.ndarray]:
    """Applies the scaling method."""
    if method == "raw":
        return X_train.copy(), X_test.copy()
    elif method == "method3a":
        # Kind of adjustment, fitted specifically on S domain, not the entire train.
        scaler = StandardScaler().fit(X_train[is_s_train])
        return scaler.transform(X_train), scaler.transform(X_test)
    elif method == "method3b":
        # Fit on train, transform both, canonical ML method.
        scaler = StandardScaler().fit(X_train)
        return scaler.transform(X_train), scaler.transform(X_test)
    elif method == "method1a":
        # Each dataset owns scaler
        X_train_scaled_s = StandardScaler().fit(X_train[is_s_train]).transform(X_train[is_s_train])
        X_train_scaled_r = StandardScaler().fit(X_train[~is_s_train]).transform(X_train[~is_s_train])

        X_test_scaled_s = StandardScaler().fit(X_test[is_s_test]).transform(X_test[is_s_test])
        X_test_scaled_r = StandardScaler().fit(X_test[~is_s_test]).transform(X_test[~is_s_test])

        X_train_scaled = np.copy(X_train)
        X_test_scaled = np.copy(X_test)

        X_train_scaled[is_s_train] = X_train_scaled_s
        X_train_scaled[~is_s_train] = X_train_scaled_r

        X_test_scaled[is_s_test] = X_test_scaled_s
        X_test_scaled[~is_s_test] = X_test_scaled_r
        
        return X_train_scaled, X_test_scaled
    elif method == "method1b":
        # Separate scaler for Train and Test
        X_train_scaled = StandardScaler().fit(X_train).transform(X_train)
        X_test_scaled = StandardScaler().fit(X_test).transform(X_test)
        return X_train_scaled, X_test_scaled
    elif method == "method1c":
        # Separate scaler per domain
        scaler_s = StandardScaler().fit(X_train[is_s_train])
        scaler_r = StandardScaler().fit(X_train[~is_s_train])

        X_train_scaled_s = scaler_s.transform(X_train[is_s_train])
        X_train_scaled_r = scaler_r.transform(X_train[~is_s_train])

        X_test_scaled_s = scaler_s.transform(X_test[is_s_test])
        X_test_scaled_r = scaler_r.transform(X_test[~is_s_test])

        X_train_scaled = np.copy(X_train)
        X_test_scaled = np.copy(X_test)

        X_train_scaled[is_s_train] = X_train_scaled_s
        X_train_scaled[~is_s_train] = X_train_scaled_r

        X_test_scaled[is_s_test] = X_test_scaled_s
        X_test_scaled[~is_s_test] = X_test_scaled_r

        return X_train_scaled, X_test_scaled

    raise ValueError(f"Unknown scaling method: {method}")

def _undersample_train(x_train: np.ndarray, y_train: np.ndarray, random_state: int) -> tuple[np.ndarray, np.ndarray]:
    """Basic undersampling."""
    rng = np.random.default_rng(random_state)
    idx_0 = np.where(y_train == 0)[0]
    idx_1 = np.where(y_train == 1)[0]
    
    n_minor = min(len(idx_0), len(idx_1))
    
    idx_0_sampled = rng.choice(idx_0, n_minor, replace=False)
    idx_1_sampled = rng.choice(idx_1, n_minor, replace=False)
    
    idx_balanced = np.concatenate([idx_0_sampled, idx_1_sampled])
    rng.shuffle(idx_balanced)
    
    return x_train[idx_balanced], y_train[idx_balanced]

def train_domain_classifiers_cv(
    df_s: pd.DataFrame,
    df_r: pd.DataFrame,
    feature_cols: list[str],
    estimators: dict[str, BaseEstimator],
    cv_folds: int = 5,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, np.ndarray], pd.DataFrame]:
    """Train and evaluate domain classifiers using Cross-Validation.
    Returns: (df_results_balanced, df_results_undersample, confusion_matrices, df_top_fp_files)
    """
    n_s, n_r = len(df_s), len(df_r)
    
    X_all = np.vstack([
        df_s[feature_cols].to_numpy(dtype=float), 
        df_r[feature_cols].to_numpy(dtype=float)
    ])
    y_all = np.concatenate([np.zeros(n_s, dtype=int), np.ones(n_r, dtype=int)])
    is_s_all = np.concatenate([np.ones(n_s, dtype=bool), np.zeros(n_r, dtype=bool)])
    
    file_numbers_all = np.concatenate([
        df_s[FILE_NUMBER_COL].to_numpy(), 
        df_r[FILE_NUMBER_COL].to_numpy()
    ])

    rows = []
    confusion_matrices: dict[str, np.ndarray] = {}
    fp_records = []
    
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    
    for method in ("raw", "method1a", "method1b", "method1c", "method3a", "method3b"):
        for train_policy in ("undersample", "balanced"):
            for algorithm_name, base_estimator in estimators.items():
                
                y_pred_oof = np.zeros_like(y_all, dtype=int)
                
                for train_idx, test_idx in skf.split(X_all, y_all):
                    X_train_raw, X_test_raw = X_all[train_idx], X_all[test_idx]
                    y_train, _ = y_all[train_idx], y_all[test_idx]
                    is_s_train, is_s_test = is_s_all[train_idx], is_s_all[test_idx]
                    
                    X_train_scaled, X_test_scaled = _apply_scaling(
                        X_train_raw, X_test_raw, is_s_train, is_s_test, method
                    )
                    
                    model = clone(base_estimator)
                    if train_policy == "undersample":
                        X_fit, y_fit = _undersample_train(X_train_scaled, y_train, random_state)
                    else:
                        X_fit, y_fit = X_train_scaled, y_train
                        try:
                            model.set_params(class_weight="balanced")
                        except ValueError:
                            print(f"Estimator {algorithm_name} doesn't support class_weight 'balanced'")
                            pass
                    
                    model.fit(X_fit, y_fit)
                    y_pred_oof[test_idx] = model.predict(X_test_scaled)
                
                key = f"{method}_{algorithm_name}_{train_policy}"
                confusion_matrices[key] = confusion_matrix(y_all, y_pred_oof)
                
                metric_name = "balanced_accuracy" if train_policy == "balanced" else "accuracy"
                metric_func = balanced_accuracy_score if metric_name == "balanced_accuracy" else accuracy_score
                metric_value = float(metric_func(y_all, y_pred_oof))
                
                rows.append({
                    "scaling": method,
                    "algorithm": algorithm_name,
                    "train_policy": train_policy,
                    "metric_name": metric_name,
                    "metric_value": metric_value,
                    "cv_folds": cv_folds,
                    "random_state": random_state,
                })
                
                fp_mask = (y_all == 0) & (y_pred_oof == 1) & (file_numbers_all >= 0)
                fp_files = file_numbers_all[fp_mask]
                
                if len(fp_files) > 0:
                    top_files = pd.Series(fp_files).value_counts().head(10)
                    for file_num, count in top_files.items():
                        fp_records.append({
                            "Configuration": key,
                            FILE_NUMBER_COL: file_num,
                            "FP_Count": count
                        })

    df_all_results = pd.DataFrame(rows)
    df_balanced = df_all_results[df_all_results["train_policy"] == "balanced"].reset_index(drop=True)
    df_undersample = df_all_results[df_all_results["train_policy"] == "undersample"].reset_index(drop=True)
    
    df_top_fp = pd.DataFrame(fp_records)
    
    return df_balanced, df_undersample, confusion_matrices, df_top_fp


def plot_confusion_grid(
    confusion_matrices: dict[str, np.ndarray],
    method: str,
    train_policy: str,
) -> Figure:
    """Plot confusion matrices for a specific method and policy."""
    keys = [k for k in sorted(confusion_matrices) if k.startswith(f"{method}_") and k.endswith(f"_{train_policy}")]
    
    n_keys = len(keys)
    cols = min(3, n_keys)
    rows = max(1, (n_keys + cols - 1) // cols)
    
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes_flat = np.atleast_1d(axes).flatten()
    
    for ax, key in zip(axes_flat, keys, strict=False):
        ConfusionMatrixDisplay(confusion_matrices[key], display_labels=[S_DOMAIN, R_DOMAIN]).plot(
            ax=ax, colorbar=False, cmap="Blues"
        )
        algorithm_name = key.replace(f"{method}_", "").replace(f"_{train_policy}", "")
        ax.set_title(algorithm_name, fontsize=10, fontweight="bold")
        
    for ax in axes_flat[len(keys):]:
        ax.axis("off")
        
    fig.suptitle(f"Domain classifiers CV-OOF ({method} / {train_policy})", fontsize=14, y=1.02)
    fig.tight_layout()
    return fig

In [ ]:
estimators_dict = {
    "LR_default_max_iter_2000": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "SVC_as_in_paper": SVC(**SVC_MODEL_PARAMS, random_state=RANDOM_STATE),
    "RF_default": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
}

df_balanced, df_undersample, confusion_matrices, df_top_fp = train_domain_classifiers_cv(
    df_s=sorted_s,
    df_r=sorted_r, 
    feature_cols=feature_cols,
    estimators=estimators_dict,
    cv_folds=CLASSIFIER_CV_FOLDS,
    random_state=RANDOM_STATE
)

cm_out_dir = OUT_DIR / "confusion_matrices"
cm_out_dir.mkdir(parents=True, exist_ok=True)

methods = ["raw", "method1a", "method1b", "method1c", "method3a", "method3b"]
policies = ["undersample", "balanced"]

for policy in policies:
    for method in methods:
        if any(k.startswith(f"{method}_") and k.endswith(f"_{policy}") for k in confusion_matrices):
            
            fig = plot_confusion_grid(
                confusion_matrices=confusion_matrices,
                method=method,
                train_policy=policy,
            )
            
            file_name = f"fig_cm_{method}_{policy}.svg"
            fig.savefig(
                cm_out_dir / file_name,
                format="svg",
                bbox_inches="tight",
                facecolor="white",
                dpi=1200,
            )
            plt.close(fig)

pivot_balanced = df_balanced.pivot(
    index="algorithm", 
    columns="scaling", 
    values="metric_value"
).round(2)

pivot_undersample = df_undersample.pivot(
    index="algorithm", 
    columns="scaling", 
    values="metric_value"
).round(2)

pivot_balanced.to_csv(OUT_DIR / "classifiers_metrics_balanced.csv")
pivot_undersample.to_csv(OUT_DIR / "classifiers_metrics_undersample.csv")
df_top_fp.to_csv(OUT_DIR / "classifiers_top_10_fp_files.csv", index=False)

print("BALANCED POLICY (Balanced Accuracy)")
display(pivot_balanced)

print("\nUNDERSAMPLE POLICY (Accuracy)")
display(pivot_undersample)

print("\nTOP 10 FP files")
display(df_top_fp.head(10))